# Activity: Simulated Annealing for Function Optimization
In this activity, you will implement the Simulated Annealing algorithm to solve a classic problem in quantiative finance: the minimum variance portfolio allocation problem.

> __Learning Objectives:__
> 
> Fill me in

This is fun application of the Simulated Annealing algorithm to a real-world problem in finance. So, let's get started!

___

## Background: Minimum Variance Portfolio Allocation Problem
The goal of minimum variance portfolio allocation is to find the optimal weights $\mathbf{w}$ that minimize the portfolio risk for a given level of expected return, or equivalently, maximize the expected return for a given level of risk. This is typically done by solving a __constrained quadratic programming__ problem. 

### Portfolio Reward
The reward of a portfolio is measured by its expected growth (return), which is the weighted average of the expected growth rates (returns) of the individual assets in the portfolio. 

Suppose we have a portfolio $\mathcal{P}$ consisting of $M$ assets. Let $w_i\in\mathbb{R}_{\geq 0}$ be the weight of asset $i$ in the portfolio (i.e., the dollar fraction of the total portfolio value invested in asset $i$), and let $\mathbb{E}[g_i]$ be the expected growth rate (return) of asset $i$. Then, the expected growth rate (return) of the portfolio, denoted as $\mathbb{E}[g_{\mathcal{P}}]$, is given by:
$$
\mathbb{E}[g_{\mathcal{P}}] = \sum_{i=1}^{M} w_i \mathbb{E}[g_i]\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]
$$
where $M$ is the total number of assets in the portfolio, i.e., $|\mathcal{P}| = M$, the weight vector is $\mathbf{w}^{\top} = [w_1, w_2, \dots, w_M]$, the sum of weights is one, and $\mathbb{E}[\mathbf{g}] = [\mathbb{E}[g_1], \mathbb{E}[g_2], \dots, \mathbb{E}[g_M]]^{\top}$ is the vector of expected growth rates (returns) of the individual assets.

### Portfolio Risk
The risk of a portfolio is (typically) measured by its variance (or standard deviation) of growth rates (returns), which takes into account the variances of the individual assets as well as the covariances between them. The portfolio variance written in terms of growth rates is given by:
$$
\text{Var}(g_{\mathcal{P}}) = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(g_i, g_j)\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}
$$
where $\text{Cov}(g_i, g_j)$ is the covariance between the growth rates of assets $i$ and $j$, and $\mathbf{\Sigma}_{g}$ is the covariance matrix of asset growth rates, defined as:
$$
\mathbf{\Sigma}_{g} =
\begin{bmatrix}
\text{Var}(g_1) & \text{Cov}(g_1, g_2) & \cdots & \text{Cov}(g_1, g_M) \\
\text{Cov}(g_2, g_1) & \text{Var}(g_2) & \cdots & \text{Cov}(g_2, g_M) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(g_M, g_1) & \text{Cov}(g_M, g_2) & \cdots & \text{Var}(g_M)
\end{bmatrix}
$$

### Optimization Problem Formulation
Let's consider the case when we have a portfolio $\mathcal{P}$ consisting of $M$ __risky assets__, i.e., only equity, ETFs (or potentially derivatives) but no fixed income assets. In this case, we can formulate the optimal weights optimization problem (written in terms of growth rate) as:

$$
\boxed{
\begin{align*}
\text{minimize}~\text{Var}(g_{\mathcal{P}}) &= \sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_{i}w_{j}\underbrace{\text{Cov}\left(g_{i},g_{j}\right)}_{= \sigma_{i}\sigma_{j}\rho_{ij}}\quad{\Longleftrightarrow\mathbf{w}^\top \mathbf{\Sigma}_{g} \mathbf{w}} \\
\text{subject to}~\mathbb{E}(g_{\mathcal{P}})& =  \sum_{i\in\mathcal{P}}w_{i}\;\mathbb{E}(g_{i})= R^{*}\quad\Longleftrightarrow\mathbf{w}^\top \mathbb{E}(\mathbf{g}) = R^{*} \\
\sum_{i\in\mathcal{P}}w_{i} & =  1 \\
w_{i} & \geq  0\quad\forall{i}\in\mathcal{P}
\end{align*}}
$$
The term $R^{*}$ is the target annualized growth rate (return) for portfolio $\mathcal{P}$ specified by the investor. The $w_{i}\geq{0}~\forall{i}\in\mathcal{P}$ and the summation-to-unity constraints forbid short selling (borrowing). If short selling (borrowing) is allowed, these constraints can be relaxed.

This is a __constrained quadratic programming__ problem, which can be solved using various optimization techniques, including __Simulated Annealing__.

___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's setup our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Data
We gathered a daily open-high-low-close `dataset` for each firm in the [S&P500](https://en.wikipedia.org/wiki/S%26P_500) from `01-03-2014` until `12-31-2024`, along with data for a few exchange-traded funds and volatility products during that time. 

Let's load the `original_dataset::DataFrame` by calling [the `MyTrainingMarketDataSet()` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyTrainingMarketDataSet) and remove firms that do not have the maximum number of trading days. The cleaned dataset $\mathcal{D}$ will be stored in the `dataset` variable.

In [2]:
original_dataset = MyTrainingMarketDataSet() |> x-> x["dataset"];

Not all tickers in our dataset have the maximum number of trading days for various reasons, e.g., acquisition or de-listing events. Let's collect only those tickers with the maximum number of trading days.

First, let's compute the number of records for a firm that we know has a maximum value, e.g., `AAPL`, and save that value in the `maximum_number_trading_days::Int64` variable:

In [3]:
maximum_number_trading_days = original_dataset["AAPL"] |> nrow # nrow? (check out: DataFrames.jl)

2767

Now, let's iterate through our data and collect only tickers with `maximum_number_trading_days` records. Save that data in the `dataset::Dict{String,DataFrame}` variable:

In [4]:
dataset = let

    # initialize -
    dataset = Dict{String, DataFrame}();

    # iterate through the dictionary; we can't guarantee a particular order
    for (ticker, data) ∈ original_dataset  # we get each (K, V) pair!
        if (nrow(data) == maximum_number_trading_days) # what is this doing?
            dataset[ticker] = data;
        end
    end
    dataset; # return
end;

Finally, let's get a list of the firms in our cleaned up dataset (and sort them alphabetically). We store the sorted firm ticker symbols in the `list_of_tickers::Array{String,1}` variable.

In [5]:
list_of_tickers = keys(dataset) |> collect |> sort; # list of firm "ticker" symbols in alphabetical order

___

## Task 1: Compute the growth rate matrix
In this task, we compute the growth rate array which contains, for each day and each firm in our dataset, the value of the growth rate between time $j$ and $j-1$. 

>  __Continuously Compounded Growth Rate (CCGR)__
>
> Let's assume a model of the share price of firm $i$ is governed by an expression of the form:
>$$
\begin{align*}
S^{(i)}_{j} &= S^{(i)}_{j-1}\;\exp\left(g^{(i)}_{j,j-1}\Delta{t}_{j}\right)
\end{align*}
> $$
> where $S^{(i)}_{j-1}$ denotes the share price of firm $i$ at time index $j-1$, $S^{(i)}_{j}$ denotes the share price of firm $i$ at time index $j$, and $\Delta{t}_{j} = t_{j} - t_{j-1}$ denotes the length of a time step (units: years) between time index $j-1$ and $j$. The value we are going to estimate is the growth rate $g^{(i)}_{j,j-1}$ (units: inverse years) for each firm $i$, and each time step in the dataset.

We've implemented [the `log_growth_rate(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.log_growth_matrix) which takes the cleaned up dataset, and a list of ticker symbols, and returns the growth rate array. Each row of the growth rate array is a time step, while each column corresponds to a firm from the `list_of_tickers::Array{String,1}` array.

In [6]:
growth_rate_array = let

    # initialize -
    Δt = (1/252); # time-step one-day in units of years (trading year is 252 days)
    r̄ = 0.0; # assume the risk-free rate is 0

    # compute the growth matrix -
    growth_rate_array = log_growth_matrix(dataset, list_of_tickers, Δt = Δt, 
        risk_free_rate = r̄); # other optional parameters are at their defaults

    growth_rate_array; # return
end

2766×424 Matrix{Float64}:
 -0.877554    6.28105    -2.87097     …  -0.755391   0.245894  -1.00527
  2.81626     1.07149     1.39239         2.13832   -0.80279    0.986468
  3.31305     0.855597    0.00536803      0.109877   1.191     -2.58144
  0.646425   17.2599      1.69215         0.274716   3.1593    -0.368228
  1.81609     2.57961     3.31924         0.621677  -2.1687     4.40309
  0.61383    -3.96384    -0.79278     …  -0.862739  -1.90977   -3.11624
  2.86071    -0.483751    4.84573         1.7657    -1.77685   -1.0896
  2.04671     1.0135      1.90809         1.67597    4.44984   -0.137819
  1.31289     1.67413     0.107259       -1.50708   -2.13696    1.43784
  1.22016     6.12957     0.932578       -1.53202    2.87784   -1.43626
  ⋮                                   ⋱                        
 -4.36889     3.84443    -2.37452        -4.26011   -9.17906   -3.94641
 -2.51182    -2.60891   -10.1209         -3.03895   -7.07468   -7.14019
  2.21355     4.15066     7.27678         3.

> **Growth Rate Matrix Structure**
>
> The `growth_rate_array` is a matrix $\mathbf{G} \in \mathbb{R}^{m \times n}$ where each **row** represents a trading day (time step) in our dataset, each **column** represents a firm from the S&P500, and each **element** $G_{i,j}$ contains the continuously compounded growth rate for firm $j$ on day $i$.

The matrix has 424 firms (columns) and there are $T-1$ = 2,766 trading days (rows), capturing the daily growth rate dynamics of the S&P500 components from 2014 to 2024. 

Let's approximate the expected growth rate vector $\mathbb{E}[\mathbf{g}]$ and the covariance matrix $\mathbf{\Sigma}_{g}$ using the `growth_rate_array`.

The expected growth rate vector $\mathbb{E}[\mathbf{g}]$ can be approximated by computing the mean of each column in the `growth_rate_array.` Let's save the expected growth rate vector in the `ḡ::Array{Float64,1}` variable:

In [14]:
ḡ = mean(growth_rate_array, dims = 1) |> vec # expected growth rate vector (units: inverse years)

424-element Vector{Float64}:
  0.10874477980244975
 -0.03703243517086077
 -0.0799094577252642
  0.23280114847444194
  0.11114647191084896
  0.09794979247859234
  0.1333019961724274
  0.1835705859232435
  0.1326786226774918
  0.01415870983447601
  ⋮
 -0.0751151192174708
  0.08180190853490005
  0.006782878039138938
 -0.08549763123281434
  0.11052104541338774
  0.052025477386285615
  0.1796939974365368
  0.05468756164364767
  0.14764000690210635

Fill me in.

In [ ]:
Σ̂ = let
    Δt = (1/252); # time-step one-day in units of years (trading year is 252 days)
    R = growth_rate_array; # shorthand
    Σ = cov(R, dims=1)*(Δt); # covariance matrix (units: dimensionless)
end

424×424 Matrix{Float64}:
 0.0535771   0.032397    0.0212982  …  0.0355662   0.0280938   0.0266666
 0.032397    0.207093    0.0431437     0.051429    0.0689041   0.0280481
 0.0212982   0.0431437   0.117391      0.0308539   0.037867    0.019717
 0.0229702   0.0312627   0.0163174     0.032633    0.0201827   0.022609
 0.0179206   0.0176175   0.0158407     0.0157326   0.0168026   0.0185472
 0.0244078   0.0195811   0.0145372  …  0.023159    0.0167416   0.0221966
 0.025643    0.0334402   0.0218931     0.0313218   0.0280694   0.023824
 0.0294242   0.02952     0.0185233     0.0369478   0.0198333   0.0262802
 0.0297618   0.0458278   0.0231343     0.0433011   0.0335821   0.0233085
 0.0169025   0.0325409   0.0206063     0.0219846   0.0327002   0.0131991
 ⋮                                  ⋱                          
 0.0339199   0.0921199   0.035255   …  0.0503196   0.0581754   0.0308858
 0.00888225  0.00814595  0.0118987     0.00616705  0.00592554  0.0112662
 0.0161836   0.0376862   0.0190661    

## Task 2: Solve the optimal allocation problem using Simulated Annealing
In this task, we will implement the Simulated Annealing algorithm to solve the minimum variance portfolio allocation problem formulated above. This is an example of a constrained optimization problem, which we can solve using Simulated Annealing by incorporating the constraints into the objective function using penalty terms.

In the general case, we can write the (augmented) objective function $P_{\mu,\rho}(x)$:
$$
\begin{align*}
    \min_{x\in\mathbb{R}^n}\;P_{\mu,\rho}(x)\;&=f(x)\;-\;\mu\sum_{i=1}^m\ln\bigl(-\,g_i(x)\bigr)\;+\;\frac{1}{2\rho}\sum_{j=1}^p 
    \bigl[h_j(x)\bigr]^2,\quad\text{where}\quad\mu>0,\;\rho>0\\
\end{align*}
$$
where $f(x)$ is the objective function, $g_i(x)$ are the inequality constraints, and $h_j(x)$ are the equality constraints. For our allocation problem, we have both equality and inequality constraints. The equality constraints are $\mathbf{w}^\top \mathbb{E}(\mathbf{g}) = R^{*}$ and $\sum_{i\in\mathcal{P}}w_{i} = 1$, while the inequality constraints are $w_{i} \geq 0~\forall{i}\in\mathcal{P}$. 

Putting this all together, we can write the augmented objective function for our allocation problem as:
$$
\boxed{
\begin{align*}
    \min_{\mathbf{w}\in\mathbb{R}^M}\;P_{\mu,\rho}(\mathbf{w})\;&=\;\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}\;-\;\mu\sum_{i=1}^M\ln\bigl(w_i\bigr)\;+\;\frac{1}{2\rho}\Bigr[\bigl(\mathbf{w}^\top ḡ - R^{*}\bigr)^2\;+\;\bigl(1-\sum_{i=1}^M w_i\bigr)^2\Bigl]\quad\blacksquare\\
\end{align*}}
$$

### Implementation
Fill me in.

In [15]:
my_ticker_array = ["MSFT", "AMD", "NVDA", "PG", "GS", "WMT"]; # select some tickers -